In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
import xgboost as xgb 
from xgboost.sklearn import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

from datetime import date
from tqdm import tqdm
import os 

# Loading Data

In [ ]:
wq_data = pd.read_csv("complete_data.csv")

# Model Building

In [ ]:
df = wq_data.copy()

In [ ]:
wq_data = df.copy()

In [ ]:
wq_data.columns

In [ ]:
# Retaining only the columns for swir22, NDMI, MNDWI, pet, Total Alkalinity, Electrical Conductance and Dissolved Reactive Phosphorus Index in the dataset.
wq_data_TA = wq_data[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 
        'green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI', 
       'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 'Month_sin', 'Month_cos',
       'season_autumn', 'season_spring', 'season_summer', 'season_winter'
       ]].copy()

# Retaining only the columns for swir22, NDMI, MNDWI, pet, Total Alkalinity, Electrical Conductance and Dissolved Reactive Phosphorus Index in the dataset.
wq_data_EC = wq_data[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 
        'green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI', 
       'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 'Month_sin', 'Month_cos',
       'season_autumn', 'season_spring', 'season_summer', 'season_winter']].copy()
        
wq_data_DRP = wq_data[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 
        'green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI', 
       'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 'Month_sin', 'Month_cos',
       'season_autumn', 'season_spring', 'season_summer', 'season_winter']].copy()






# XGBoost

In [ ]:
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train):
    model = XGBRegressor()
    model.fit(X_train_scaled, y_train)
    return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n{dataset_name} Evaluation:")
    print(f"R²: {r2:.3f}")
    print(f"RMSE: {rmse:.3f}")
    return y_pred, r2, rmse

# Model Workflow (Pipeline)

In [ ]:
def run_pipeline(X, y, param_name="Parameter"):
    print(f"\n{'='*60}")
    print(f"Training Model for {param_name}")
    print(f"{'='*60}")
    
    # Split data
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # Scale
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    
    # Train
    model = train_model(X_train_scaled, y_train)
    
    # Evaluate (in-sample)
    y_train_pred, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")
    
    # Evaluate (out-sample)
    y_test_pred, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, "Test")
    
    # Return summary
    results = {
        "Parameter": param_name,
        "R2_Train": r2_train,
        "RMSE_Train": rmse_train,
        "R2_Test": r2_test,
        "RMSE_Test": rmse_test
    }


    
    return model, scaler, pd.DataFrame([results])

In [ ]:
X_TA = wq_data_TA.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
X_EC = wq_data_EC.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
X_DRP = wq_data_DRP.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])


y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

model_TA, scaler_TA, results_TA = run_pipeline(X_TA, y_TA, "Total Alkalinity")
model_EC, scaler_EC, results_EC = run_pipeline(X_EC, y_EC, "Electrical Conductance")
model_DRP, scaler_DRP, results_DRP = run_pipeline(X_DRP, y_DRP, "Dissolved Reactive Phosphorus")

In [ ]:
def get_feature_importance(model, X, model_name):
    
    feature_imp_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    
    print(f"\nFeature Importance for {model_name}")
    print(feature_imp_df)
    
    return feature_imp_df


imp_TA = get_feature_importance(model_TA, X_TA, "Total Alkalinity")
imp_EC = get_feature_importance(model_EC, X_EC, "Electrical Conductance")
imp_DRP = get_feature_importance(model_DRP, X_DRP, "Dissolved Reactive Phosphorus")

# Model Performance

In [ ]:
results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
results_summary

# Tuning Model Workflow

In [ ]:
# Using GridSearchCV

from sklearn.model_selection import GridSearchCV

def tuning (X_train_scaled, y_train):

    param_grid = {
    'colsample_bytree': [0.3, 0.6],   
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth' : [2, 4, 6]
    }

    xgb = XGBRegressor()

    grid_search = GridSearchCV(
    estimator= XGBRegressor(colsample_bytree=0.8, subsample=0.8),
    param_grid = param_grid,
    cv = 5,
    scoring = 'r2'
    )

    model_tuned = grid_search.fit(X_train_scaled, y_train)
    return model_tuned



In [ ]:
def run_tuning(X, y, param_name="Parameter"):
    print(f"\n{'='*60}")
    print(f"Training Model for {param_name}")
    print(f"{'='*60}")
    
    # Split data
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # Scale
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    
    # Train
    model = tuning(X_train_scaled, y_train)
    
    # Evaluate (in-sample)
    y_train_pred, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")
    
    # Evaluate (out-sample)
    y_test_pred, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, "Test")
    
    # Return summary
    results_tuned = {
        "Parameter": param_name,
        "R2_Train": r2_train,
        "RMSE_Train": rmse_train,
        "R2_Test": r2_test,
        "RMSE_Test": rmse_test
    }
    
    return model, scaler, pd.DataFrame([results_tuned])

In [ ]:
model_TA_tuned, scaler_TA, results_TA_tuned = run_tuning(X_TA, y_TA, "Total Alkalinity Tuned")
model_EC_tuned, scaler_EC, results_EC_tuned = run_tuning(X_EC, y_EC, "Electrical Conductance Tuned")
model_DRP_tuned, scaler_DRP, results_DRP_tuned = run_tuning(X_DRP, y_DRP, "Dissolved Reactive Phosphorus Tuned")

results_summary_tuned = pd.concat([results_TA_tuned, results_EC_tuned, results_DRP_tuned], ignore_index=True)
results_summary_tuned

In [ ]:
results_summary_tuned = pd.concat([results_TA_tuned, results_EC_tuned, results_DRP_tuned], ignore_index=True)
results = pd.concat([results_summary, results_summary_tuned])
results

# Submission file - Validation data

In [ ]:
val_data = pd.read_csv("complete_val_data.csv")

In [ ]:
val_data.columns

In [ ]:
# Retaining only the columns for swir22, NDMI, MNDWI, pet, Total Alkalinity, Electrical Conductance and Dissolved Reactive Phosphorus Index in the dataset.
val_data_TA = val_data[['green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI', 
       'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 'Month_sin', 'Month_cos',
       'season_autumn', 'season_spring', 'season_summer', 'season_winter'
       ]].copy()

# Retaining only the columns for swir22, NDMI, MNDWI, pet, Total Alkalinity, Electrical Conductance and Dissolved Reactive Phosphorus Index in the dataset.
val_data_EC = val_data[['green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI', 
       'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 'Month_sin', 'Month_cos',
       'season_autumn', 'season_spring', 'season_summer', 'season_winter']].copy()
        
val_data_DRP = val_data[['green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI', 
       'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 'Month_sin', 'Month_cos',
       'season_autumn', 'season_spring', 'season_summer', 'season_winter']].copy()


In [ ]:
# --- Predicting for Total Alkalinity ---
X_sub_scaled_TA = scaler_TA.transform(val_data_TA)
pred_TA_submission = model_TA.predict(X_sub_scaled_TA)

# --- Predicting for Electrical Conductance ---
X_sub_scaled_EC = scaler_EC.transform(val_data_EC)
pred_EC_submission = model_EC.predict(X_sub_scaled_EC)

# --- Predicting for Dissolved Reactive Phosphorus ---
X_sub_scaled_DRP = scaler_DRP.transform(val_data_DRP)
pred_DRP_submission = model_DRP.predict(X_sub_scaled_DRP)

In [ ]:
# --- Predicting for Total Alkalinity ---
X_sub_scaled_TA = scaler_TA.transform(val_data_TA)
pred_TA_tuned = model_TA_tuned.predict(X_sub_scaled_TA)

# --- Predicting for Electrical Conductance ---
X_sub_scaled_EC = scaler_EC.transform(val_data_EC)
pred_EC_tuned = model_EC_tuned.predict(X_sub_scaled_EC)

# --- Predicting for Dissolved Reactive Phosphorus ---
X_sub_scaled_DRP = scaler_DRP.transform(val_data_DRP)
pred_DRP_tuned = model_DRP_tuned.predict(X_sub_scaled_DRP)

In [ ]:
submission_df = pd.DataFrame({
    'Longitude': val_data['Longitude'].values,
    'Latitude': val_data['Latitude'].values,
    'Sample Date': val_data['Sample Date'].values,
    'Total Alkalinity': pred_TA_submission,
    'Electrical Conductance': pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission
})

In [ ]:
#Displaying the sample submission dataframe
#display(submission_df.head())

In [ ]:
#Dumping the predictions into a csv file.
submission_df.to_csv("/tmp/submission_XGBoost.csv",index = False)

session.sql("""
    PUT file:///tmp/submission_XGBoost.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print("File saved! Refresh the browser to see the files in the sidebar")

In [ ]:
# ── Load files ──────────────────────────────────────────────────────────────
actual = pd.read_csv('submission_extracted.csv')   # ground truth
pred   = pd.read_csv('submission_XGBoost.csv')     # model predictions
 
# ── Standardise coordinates & dates for merging ─────────────────────────────
for df in [actual, pred]:
    df['lat_r']      = df['Latitude'].round(3)
    df['lon_r']      = df['Longitude'].round(3)
    df['Sample Date'] = pd.to_datetime(df['Sample Date'])
 
# ── Merge on location + date ─────────────────────────────────────────────────
merged = actual.merge(pred, on=['lat_r', 'lon_r', 'Sample Date'],
                      suffixes=('_actual', '_pred'))
print(f"Rows matched: {len(merged)} / {len(actual)}\n")
 


In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
targets = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus',
]
 
results = []
for t in targets:
    y_true = merged[f'{t}_actual']
    y_pred = merged[f'{t}_pred']
 
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
 
    results.append({'Target': t, 'R²': round(r2, 4),
                     'RMSE': round(rmse, 4)})
 
    print(f"{t}")
    print(f"  R²   = {r2:.4f}")
    print(f"  RMSE = {rmse:.4f}\n")
 
# ── Overall mean R² (competition metric) ────────────────────────────────────
mean_r2 = np.mean([r['R²'] for r in results])
print(f"Mean R² (all targets): {mean_r2:.4f}")

In [ ]:
Total Alkalinity
  R²   = 0.2673
  RMSE = 91.4510

Electrical Conductance
  R²   = 0.0441
  RMSE = 990.7668

Dissolved Reactive Phosphorus
  R²   = 0.3107
  RMSE = 24.0618

Mean R² (all targets): 0.2074